**1**.**Parsing** **of** **Messages**\
**2**.**The** **Real** **code**



In [8]:
with open('/content/hostel_bois.txt', 'r', encoding='utf-8') as f:
 content = f.read()

lines=content.splitlines()

messages=[]

for i in lines:
  if " - " not in i:
    continue
  parts=i.split(" - ")
  timestamp=parts[0]
  date_time = timestamp.split(", ")

  date=date_time[0]
  time=date_time[1]

  if ":" in parts[1]:
    sender_message=parts[1].split(": ",1)
    sender = sender_message[0]
    message=sender_message[1]
    print(sender_message)

    data={
    "date":date,
    "time": time,
    "sender":sender,
    "message":message
    }

    messages.append(data)
  else:
    print("system message")
print(messages[:3])
print(len(lines))
print(len(messages))


system message
system message
system message
['Rahul', 'scene fix']
['Rahul', 'haan']
['Rahul', 'kya scene']
['Rahul', 'abhi free hai?']
['Rahul', 'abey']
['Rahul', 'bhai sun na']
['Priya', 'I have extra notes if anyone wants']
['Priya', 'Anyone needs help with anything?']
['Priya', "Don't forget to call home guys"]
['Rahul', 'ja yaar']
['Rahul', 'kya scene hai bhai']
['Rahul', 'bhai chai pite hai']
['Priya', 'Did you eat anything today?']
['Priya', 'Everyone doing okay with studies?']
['Karan', "Today's class was honestly the most interesting one we have had this semester, the professor went off topic completely and started telling us about his time in IIT during the eighties, how they used to study under streetlights because there was no electricity in the hostel and they would walk three kilometers to the library every day just to read journals."]
['Priya', "Let's plan something this weekend"]
['Priya', 'Anyone needs help with anything?']
['Priya', 'Please eat something Aman']
['Rah

**The** **Real** **Code** **or** **Project**

In [7]:
import numpy as np
from datetime import datetime, timedelta   # only strptime and timedelta are used

WIDTH = 60   # width of every printed section

MONTHS = ['January', 'February', 'March', 'April', 'May', 'June', 'July',
          'August', 'September', 'October', 'November', 'December']
MONTHS_SHORT = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug',
                'Sep', 'Oct', 'Nov', 'Dec']

# Punctuation stripped from words / removed from text. The apostrophe is
# handled separately so that "don't" and "i'm" stay intact.
PUNCT = '.,!?;:"()[]{}-_*/\\<>@#~`|+=^&%$'


def pretty_date(d):
    """datetime -> '14 April 2024'"""
    return f"{d.day:02d} {MONTHS[d.month - 1]} {d.year}"


def short_date(d):
    """datetime -> '14 Apr'"""
    return f"{d.day:02d} {MONTHS_SHORT[d.month - 1]}"


def section(title):
    """Print the top edge of a report box:  ┌─ TITLE ─────────"""
    fill = WIDTH - len(title) - 4
    print('┌─ ' + title + ' ' + '─' * fill)


def row(text=''):
    """Print one line inside a report box."""
    print('│ ' + text)


def section_end():
    """Print the bottom edge of a report box, followed by a blank line."""
    print('└' + '─' * (WIDTH - 1))
    print()

def load_lines(path_options):
    """
    Try each path in turn (local folder first, then Colab's /content) and return the file as a list of lines.
    Returns None if no file could be opened.
    'utf-8-sig' silently drops the invisible BOM some exports start with, and errors='replace'
    means a stray bad byte can never crash the notebook.
    """
    for path in path_options:
        try:
            with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
                content = f.read()
            return content.split('\n')
        except OSError:                              # file missing, is a folder, no permission ...
            continue
    return None


def looks_like_date(line):
    """Edge case 4: a new message starts with 'NN/NN/NN' (first 8 characters)."""
    head = line[:8]
    return (len(head) == 8 and head[2] == '/' and head[5] == '/'
            and head[0:2].isdigit() and head[3:5].isdigit() and head[6:8].isdigit())


MEDIA_STUBS = ['<media omitted>', 'image omitted', 'video omitted', 'audio omitted',
               'sticker omitted', 'gif omitted', 'document omitted', 'contact card omitted']
DELETED_STUBS = ['this message was deleted', 'you deleted this message']


def classify(text):
    """Return 'media', 'deleted' or 'text' (Android and iPhone wording both recognised)."""
    clean = text.strip().lower()
    if clean in MEDIA_STUBS:
        return 'media'
    if clean in DELETED_STUBS:
        return 'deleted'
    return 'text'


def read_timestamp(stamp):
    """Turn 'DD/MM/YY, HH:MM' (or 'DD/MM/YYYY, HH:MM') into a datetime; return None if it is not a real timestamp."""
    for fmt in ['%d/%m/%y, %H:%M', '%d/%m/%Y, %H:%M']:
        try:
            return datetime.strptime(stamp, fmt)
        except ValueError:
            continue
    return None


def parse_chat(lines):
    """
    Turn raw export lines into a list of message dicts.
    Returns (messages, system_count, group_name).
    """
    messages = []
    system_count = 0
    group_name = 'Unknown Group'
    last_msg = None          # the message a continuation line would belong to

    for line in lines:
        line = line.replace('\r', '').replace('\u200e', '').replace('\ufeff', '')   # CRLF, invisible LRM mark, BOM
        if line.strip() == '':                                 # edge case 5: empty line
            continue

        # A line is a NEW entry only if it starts with NN/NN/NN and has a real 'timestamp - rest' shape.
        header = None
        if looks_like_date(line) and ' - ' in line:
            stamp, rest = line.split(' - ', 1)                 # maxsplit=1: only the FIRST ' - ' splits
            dt = read_timestamp(stamp)
            if dt is not None:
                header = (stamp, dt, rest)

        if header is None:                                     # edge case 4: multi-line message
            if last_msg is not None and last_msg['kind'] == 'text':   # junk / media stubs get no extra text
                last_msg['text'] += '\n' + line
            continue

        stamp, dt, rest = header

        if ': ' not in rest:                                   # edge case 1: system message
            system_count += 1
            last_msg = None
            marker = 'created group "'                         # grab the group name when we see it
            pos = rest.find(marker)
            if pos != -1:
                start = pos + len(marker)
                end = rest.find('"', start)
                if end != -1:
                    group_name = rest[start:end]
            continue

        sender, text = rest.split(': ', 1)                     # maxsplit=1: message text may contain ': '
        # Robustness for real chats: a system line such as  changed the subject to "A: B"
        # also contains ': ' -- a genuine sender name never has a quote mark or is very long.
        if '"' in sender or len(sender) > 40:
            system_count += 1
            last_msg = None
            continue

        last_msg = {'timestamp': stamp, 'sender': sender, 'text': text,
                    'dt': dt, 'kind': classify(text)}         # edge cases 2 & 3 are tagged via 'kind'
        messages.append(last_msg)

    messages.sort(key=lambda m: m['dt'])                       # stable sort: chronological order guaranteed
    return messages, system_count, group_name


def add_day_index(messages):
    """
    Give every message a 'day_idx' (0 = first day of the chat).
    Returns (first_date, number_of_days).
    """
    if len(messages) == 0:
        return None, 0
    for m in messages:
        # midnight of the message's day (works for 2-digit and 4-digit years alike)
        m['date'] = m['dt'] - timedelta(hours=m['dt'].hour, minutes=m['dt'].minute)
    first_date = min(m['date'] for m in messages)
    last_date = max(m['date'] for m in messages)
    for m in messages:
        m['day_idx'] = (m['date'] - first_date).days
    return first_date, (last_date - first_date).days + 1


def summarize_parse(messages, system_count, n_days):
    """Print the one-line parser summary asked for in the brief."""
    if len(messages) == 0:
        print('No messages could be parsed. Expected lines like:  12/04/24, 23:14 - Rahul: hello')
        return
    people = set()
    for m in messages:
        people.add(m['sender'])
    media = sum(1 for m in messages if m['kind'] == 'media')
    deleted = sum(1 for m in messages if m['kind'] == 'deleted')
    # NOTE: media + deleted messages ARE counted in the total (3,174) because the person did send
    # something; they are only left out of the word / message-length analysis later on.
    print(f"Successfully parsed {len(messages)} messages from {len(people)} participants "
          f"over {n_days} days, skipped {system_count} system messages, "
          f"{media} media-omitted, {deleted} deleted messages.")


# ---- run Feature 1 on the dataset ----
lines = load_lines(['hostel_bois.txt', '/content/hostel_bois.txt'])
if lines is None:
    print("hostel_bois.txt not found - upload it next to the notebook (or to /content in Colab) and re-run.")
    lines = []                                   # every later cell checks HAS_DATA, so nothing crashes

messages, system_count, group_name = parse_chat(lines)
first_date, n_days = add_day_index(messages)
HAS_DATA = len(messages) > 0

summarize_parse(messages, system_count, n_days)

def count_per_person(messages):
    """{'Rahul': 953, ...} built with a plain dict (no Counter)."""
    counts = {}
    for m in messages:
        counts[m['sender']] = counts.get(m['sender'], 0) + 1
    return counts


def rank_people(counts):
    """Names sorted by message count (high -> low); ties broken alphabetically."""
    return [name for name, _ in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))]


def person_stats(messages):
    """Per person: messages, media shares, deleted messages, words and text-message count."""
    stats = {}
    for m in messages:
        s = m['sender']
        if s not in stats:
            stats[s] = {'messages': 0, 'media': 0, 'deleted': 0, 'words': 0, 'text_msgs': 0}
        stats[s]['messages'] += 1
        if m['kind'] == 'media':
            stats[s]['media'] += 1
        elif m['kind'] == 'deleted':
            stats[s]['deleted'] += 1
        else:
            stats[s]['words'] += len(m['text'].split())
            stats[s]['text_msgs'] += 1
    return stats


def print_overview(messages, counts, people, first_date, n_days, system_count):
    last_date = first_date + timedelta(days=n_days - 1)
    section('GROUP OVERVIEW')
    row(f"Period         : {pretty_date(first_date)} to {pretty_date(last_date)} ({n_days} days)")
    row(f"Total messages : {len(messages):,}")
    row(f"Participants   : {len(people)}")
    row(f"System msgs    : {system_count} (skipped)")
    row()
    row('MESSAGES PER PERSON')
    top = counts[people[0]]
    for p in people:
        c = counts[p]
        pct = 100 * c / len(messages)
        bar_len = round(20 * c / top)
        bar = '█' * bar_len if bar_len > 0 else '.'
        row(f"  {p:<8} {bar:<20} {c:>5,} ({pct:4.1f}%)")
    section_end()


def print_person_stats(stats, people):
    section('PER-PERSON STATS')
    row(f"  {'Name':<8}{'Media':>7}{'Deleted':>9}{'Words':>9}{'Words/msg':>11}")
    for p in people:
        s = stats[p]
        avg = s['words'] / s['text_msgs'] if s['text_msgs'] > 0 else 0
        row(f"  {p:<8}{s['media']:>7}{s['deleted']:>9}{s['words']:>9,}{avg:>11.1f}")
    section_end()


if HAS_DATA:
    counts = count_per_person(messages)
    people = rank_people(counts)
    stats = person_stats(messages)

    print_overview(messages, counts, people, first_date, n_days, system_count)
    print_person_stats(stats, people)
else:
    print('No messages parsed - skipping this feature.')

def busiest_day(messages, first_date):
    """Return (date, count) of the single busiest day."""
    per_day = {}
    for m in messages:
        per_day[m['day_idx']] = per_day.get(m['day_idx'], 0) + 1
    best = max(per_day, key=lambda k: per_day[k])
    return first_date + timedelta(days=best), per_day[best]


def busiest_hour(messages, n_days):
    """Return (hour, total_count, average_per_day) for the busiest hour of the day."""
    per_hour = {}
    for m in messages:
        h = m['dt'].hour
        per_hour[h] = per_hour.get(h, 0) + 1
    best = max(per_hour, key=lambda k: per_hour[k])
    return best, per_hour[best], per_hour[best] / n_days


def print_busiest(messages, first_date, n_days):
    day, day_count = busiest_day(messages, first_date)
    hour, hour_count, hour_avg = busiest_hour(messages, n_days)
    section('PEAK ACTIVITY')
    row(f"Busiest day  : {pretty_date(day)} ({day_count} messages)")
    row(f"Busiest hour : {hour:02d}:00 - {(hour + 1) % 24:02d}:00 "
        f"({hour_count} msgs in total, avg {hour_avg:.1f} per day)")
    section_end()


if HAS_DATA:
    print_busiest(messages, first_date, n_days)
else:
    print('No messages parsed - skipping this feature.')

def build_heatmap(messages, people):
    """np.zeros matrix (people x 24 hours); increment the right cell for every message."""
    row_of = {}
    for i, p in enumerate(people):
        row_of[p] = i
    heat = np.zeros((len(people), 24), dtype=int)
    for m in messages:
        heat[row_of[m['sender']], m['dt'].hour] += 1
    return heat


def shade(value, row_max):
    """Four shading levels relative to the person's own maximum."""
    if row_max == 0 or value == 0:
        return '. '
    ratio = value / row_max
    if ratio < 0.25:
        return '. '
    if ratio < 0.50:
        return '░ '
    if ratio < 0.75:
        return '▒ '
    return '█ '


def print_heatmap(heat, people, tags=None):
    if tags is None:
        tags = {}
    section('ACTIVITY HEATMAP (messages by hour, 00-23)')
    axis = ''
    for h in range(24):
        axis += f"{h:02d}" if h % 3 == 0 else '  '
    row(f"{'':<8}{axis}")
    for i, p in enumerate(people):
        row_max = int(heat[i].max())                 # NumPy max, used for normalisation
        cells = ''
        for h in range(24):
            cells += shade(int(heat[i, h]), row_max)
        line = f"{p:<8}{cells}"
        if p in tags:
            line += '<- ' + tags[p]
        row(line)
    row()
    row('. = quiet   ░ ▒ █ = busier   (shaded per person)')
    section_end()


if HAS_DATA:
    heat = build_heatmap(messages, people)
    print_heatmap(heat, people)

    # sanity checks: row totals must equal each person's message count
    print('Row totals (NumPy sum over axis=1):', heat.sum(axis=1))
    print('Message counts                     :', [counts[p] for p in people])
    print('Matrix total:', heat.sum(), '| shape:', heat.shape)
else:
    print('No messages parsed - skipping this feature.')


STOP_TEXT = """
i me my myself we our ours you your yours he him his she her hers it its they them their
what which who whom this that these those am is are was were be been being have has had having
do does did doing a an the and but if or because as until while of at by for with about against
between into through during before after above below to from up down in out on off over under
again further then once here there when where why how all any both each few more most other
some such no nor not only own same so than too very s t can will just don should now d ll m o
re ve y would could also get got go going went one even really still
hai hain ho ka ki ke ko se ne tha thi na ha ja raha rahi
today everyone anyone everything something anything came used way three
"""
STOP_WORDS = set(STOP_TEXT.split())


def tokenize(text):
    """Lower-case, split on whitespace, strip punctuation, drop stop words and pure numbers."""
    words = []
    for raw in text.lower().replace('’', "'").split():
        w = raw.strip(PUNCT + "'")
        if w == '' or w.isdigit() or w in STOP_WORDS:
            continue
        words.append(w)
    return words


def word_counts(messages):
    """Return {word: count} for the whole group. Only 'text' messages are used (media / deleted are skipped)."""
    group = {}
    for m in messages:
        if m['kind'] != 'text':
            continue
        for w in tokenize(m['text']):
            group[w] = group.get(w, 0) + 1
    return group


def top_words(counts_dict, n):
    """Top n (word, count) pairs; ties broken alphabetically."""
    return sorted(counts_dict.items(), key=lambda kv: (-kv[1], kv[0]))[:n]


def print_top_words(group_counts):
    section("THIS GROUP'S FAVOURITE WORDS")
    top = top_words(group_counts, 10)
    if len(top) == 0:
        row('No words found.')
    else:
        biggest = top[0][1]
        for word, c in top:
            bar = '█' * max(1, round(20 * c / biggest))
            row(f"  {word:<12} {bar:<20} {c}")
    section_end()


if HAS_DATA:
    group_words = word_counts(messages)
    print_top_words(group_words)
else:
    print('No messages parsed - skipping this feature.')


def silent_streaks(messages, people, n_days):
    """
    For each person find the longest run of days with no message.
    Returns {person: {'longest', 'start', 'silent_days', 'active_days'}}  (start = day index).
    """
    active = {}
    for p in people:
        active[p] = set()
    for m in messages:
        active[m['sender']].add(m['day_idx'])

    result = {}
    for p in people:
        best_len = 0
        best_start = None
        run_len = 0
        run_start = None
        for d in range(n_days):
            if d in active[p]:
                run_len = 0                          # spoke today -> streak broken
            else:
                if run_len == 0:
                    run_start = d
                run_len += 1
                if run_len > best_len:
                    best_len = run_len
                    best_start = run_start
        result[p] = {'longest': best_len, 'start': best_start,
                     'silent_days': n_days - len(active[p]), 'active_days': len(active[p])}
    return result


def print_streaks(streaks, first_date):
    section('LONGEST SILENT STREAKS (days with zero messages)')
    ordered = sorted(streaks.items(), key=lambda kv: (-kv[1]['longest'], kv[0]))
    for p, s in ordered:
        n = s['longest']
        if n == 0:
            row(f"  {p:<8}: 0 days (never went silent)")
        else:
            start = first_date + timedelta(days=s['start'])
            end = first_date + timedelta(days=s['start'] + n - 1)
            unit = 'day' if n == 1 else 'days'
            row(f"  {p:<8}: {n} {unit} ({short_date(start)} to {short_date(end)})")
    section_end()


if HAS_DATA:
    streaks = silent_streaks(messages, people, n_days)
    print_streaks(streaks, first_date)
else:
    print('No messages parsed - skipping this feature.')


CARING_WORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
                'reminder', 'drink water', "don't forget"]
LAUGH_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
MIN_TEXT_MSGS = 50          # sample-size guard for content-based archetypes


def clean_text(text):
    """Lower-case, replace punctuation by spaces and pad with spaces so ' eat ' matches a whole word."""
    text = text.lower().replace('’', "'")
    for ch in PUNCT:
        text = text.replace(ch, ' ')
    return ' ' + text + ' '


def contains_any(cleaned, phrases):
    for ph in phrases:
        if cleaned.count(' ' + ph + ' ') > 0:
            return True
    return False


def text_of(my_msgs):
    return [m['text'].strip() for m in my_msgs if m['kind'] == 'text']


def average_bursts(messages):
    """Average length of a person's 'bursts' = messages in a row with nobody else speaking between."""
    runs = {}
    run_person = None
    run_len = 0
    for m in messages:
        if m['sender'] == run_person:
            run_len += 1
        else:
            if run_person is not None:
                if run_person not in runs:
                    runs[run_person] = []
                runs[run_person].append(run_len)
            run_person = m['sender']
            run_len = 1
    if run_person is not None:
        if run_person not in runs:
            runs[run_person] = []
        runs[run_person].append(run_len)
    return {p: sum(v) / len(v) for p, v in runs.items()}


# ---- one scoring function per archetype: f(person, my_msgs, ctx) -> (score, detail) ----

def score_spammer(person, my_msgs, ctx):
    burst = ctx['avg_burst'].get(person, 0)
    return burst / 3, f"avg {burst:.1f} msgs in a row"                # rule: avg burst > 3


def score_group_mom(person, my_msgs, ctx):
    texts = text_of(my_msgs)
    if len(texts) < MIN_TEXT_MSGS:
        return 0.0, 'not enough messages'
    hits = sum(1 for t in texts if contains_any(clean_text(t), CARING_WORDS))
    pct = 100 * hits / len(texts)
    # Brief says "highest count"; a raw count just rewards whoever talks most, so I use the
    # share of a person's messages with a caring keyword. Calibrated threshold: 30%.
    return pct / 30, f"{pct:.0f}% caring keywords"


def score_night_owl(person, my_msgs, ctx):
    r = ctx['heat_rows'][person]                                       # NumPy row from the heatmap
    total = r.sum()
    if total == 0:
        return 0.0, 'no messages'
    night = r[23] + r[0:5].sum()                                       # hours 23, 0, 1, 2, 3, 4
    pct = float(100 * night / total)
    return pct / 60, f"{pct:.1f}% msgs between 23h-04h"                # rule: > 60%


def score_storyteller(person, my_msgs, ctx):
    texts = text_of(my_msgs)
    if len(texts) < MIN_TEXT_MSGS:
        return 0.0, 'not enough messages'
    avg_words = sum(len(t.split()) for t in texts) / len(texts)
    return avg_words / 30, f"avg {avg_words:.1f} words per msg"        # rule: > 30 words


def score_drama_queen(person, my_msgs, ctx):
    texts = text_of(my_msgs)
    if len(texts) < MIN_TEXT_MSGS:
        return 0.0, 'not enough messages'
    loud = 0
    for t in texts:
        all_caps = len(t) >= 3 and t.isupper()      # isupper() ignores emojis / digits
        many_bangs = t.count('!') >= 2
        if all_caps or many_bangs:
            loud += 1
    pct = 100 * loud / len(texts)
    return pct / 30, f"{pct:.1f}% ALL-CAPS or !! msgs"                 # rule: > 30%


def score_ghost(person, my_msgs, ctx):
    silent = ctx['n_days'] - ctx['streaks'][person]['active_days']
    share = 100 * silent / ctx['n_days']
    return share / 60, f"silent on {silent} of {ctx['n_days']} days"  # rule: > 60% of days


def score_comedian(person, my_msgs, ctx):
    texts = text_of(my_msgs)
    if len(texts) < MIN_TEXT_MSGS:
        return 0.0, 'not enough messages'
    hits = sum(1 for t in texts if contains_any(clean_text(t), LAUGH_WORDS))
    pct = 100 * hits / len(texts)
    return pct / 10, f"{pct:.1f}% laugh-word msgs"                     # calibrated threshold: 10%


def score_question_master(person, my_msgs, ctx):
    texts = text_of(my_msgs)
    if len(texts) < MIN_TEXT_MSGS:
        return 0.0, 'not enough messages'
    hits = sum(1 for t in texts if t.endswith('?'))
    pct = 100 * hits / len(texts)
    return pct / 25, f"{pct:.1f}% msgs end with ?"                     # rule: > 25%


def score_punctual(person, my_msgs, ctx):
    """INVENTED (bonus): THE PAKKA PUNCTUAL ONE - online almost every day, but only at sensible hours."""
    r = ctx['heat_rows'][person]
    total = r.sum()
    if total == 0:
        return 0.0, 'no messages'
    active_share = ctx['streaks'][person]['active_days'] / ctx['n_days']
    if active_share < 0.90:
        return 0.0, 'not active enough'
    day_pct = float(100 * r[6:23].sum() / total)                       # hours 06:00 - 22:59
    return day_pct / 97, f"{day_pct:.1f}% msgs between 06h-22h"


# order matters: it is the tie-break order (earlier = wins ties)
ARCHETYPES = [
    ('THE SPAMMER', score_spammer),
    ('THE GROUP MOM', score_group_mom),
    ('THE NIGHT OWL', score_night_owl),
    ('THE STORYTELLER', score_storyteller),
    ('THE DRAMA QUEEN', score_drama_queen),
    ('THE GHOST', score_ghost),
    ('THE COMEDIAN', score_comedian),
    ('THE QUESTION MASTER', score_question_master),
    ('THE PAKKA PUNCTUAL ONE', score_punctual),
]


def build_context(messages, people, heat, streaks, n_days):
    heat_rows = {}
    for i, p in enumerate(people):
        heat_rows[p] = heat[i]
    return {'avg_burst': average_bursts(messages), 'heat_rows': heat_rows,
            'streaks': streaks, 'n_days': n_days}


def assign_archetypes(messages, people, ctx):
    """Exclusive assignment: highest score first; each person and each archetype used at most once."""
    by_person = {}
    for p in people:
        by_person[p] = []
    for m in messages:
        by_person[m['sender']].append(m)

    pairs = []
    for order, (label, fn) in enumerate(ARCHETYPES):
        for p in people:
            score, detail = fn(p, by_person[p], ctx)
            if score > 0:
                pairs.append((score, order, p, label, detail))
    pairs = sorted(pairs, key=lambda t: (-t[0], t[1], t[2]))   # tie-break: archetype order, then name

    assigned = {}
    used = set()
    for score, order, p, label, detail in pairs:
        if p in assigned or label in used:
            continue
        assigned[p] = (label, detail, score)
        used.add(label)
    for p in people:                                             # tiny chats: nothing scored
        if p not in assigned:
            assigned[p] = ('THE ENIGMA', 'not enough data', 0.0)
    return assigned


def print_archetypes(assigned, people):
    section('PERSONALITY ARCHETYPES')
    weak = False
    for p in people:
        label, detail, score = assigned[p]
        mark = ''
        if 0 < score < 1:
            mark = ' *'
            weak = True
        row(f"{p:<8} → {label} ({detail}){mark}")
    if weak:
        row()
        row('* closest fit (below the archetype threshold)')
    section_end()


if HAS_DATA:
    ctx = build_context(messages, people, heat, streaks, n_days)
    assigned = assign_archetypes(messages, people, ctx)
    print_archetypes(assigned, people)
else:
    print('No messages parsed - skipping this feature.')


def print_report(messages, system_count, group_name):
    if len(messages) == 0:
        print('No chat messages found - nothing to analyse.')
        print('Expected export lines like:  12/04/24, 23:14 - Rahul: hello')
        return

    first_date, n_days = add_day_index(messages)
    counts = count_per_person(messages)
    people = rank_people(counts)
    stats = person_stats(messages)
    heat = build_heatmap(messages, people)
    group_words = word_counts(messages)
    streaks = silent_streaks(messages, people, n_days)
    ctx = build_context(messages, people, heat, streaks, n_days)
    assigned = assign_archetypes(messages, people, ctx)

    # the heatmap gets a small tag next to the night owl
    tags = {}
    for p in people:
        if assigned[p][0] == 'THE NIGHT OWL':
            tags[p] = 'NIGHT OWL'

    print('=' * WIDTH)
    print(f' GROUPDNA REPORT — "{group_name}"')
    member_word = 'member' if len(people) == 1 else 'members'
    day_word = 'day' if n_days == 1 else 'days'
    print(f' {n_days} {day_word} • {len(messages):,} messages • {len(people)} {member_word}')
    print('=' * WIDTH)
    print()

    print_overview(messages, counts, people, first_date, n_days, system_count)
    print_person_stats(stats, people)
    print_busiest(messages, first_date, n_days)
    print_heatmap(heat, people, tags)
    print_top_words(group_words)
    print_streaks(streaks, first_date)
    print_archetypes(assigned, people)

    print('=' * WIDTH)
    print(' Generated by GroupDNA • Built with Python + NumPy')
    print('=' * WIDTH)


def run_groupdna(lines):
    """Full pipeline on the raw lines of any export. Never crashes on empty / missing / wrongly formatted input."""
    if lines is None:
        print('No file to analyse (the file could not be opened).')
        return
    messages, system_count, group_name = parse_chat(lines)
    print_report(messages, system_count, group_name)


run_groupdna(lines)

# Exception Handling , assistance has been taken from AI

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
┌─ GROUP OVERVIEW ──────────────────────────────────────────
│ Period         : 01 April 2024 to 30 May 2024 (60 days)
│ Total messages : 3,174
│ Participants   : 6
│ System msgs    : 4 (skipped)
│ 
│ MESSAGES PER PERSON
│   Rahul    ████████████████████   953 (30.0%)
│   Priya    ███████████████        718 (22.6%)
│   Neha     █████████████          635 (20.0%)
│   Aman     ██████████             490 (15.4%)
│   Karan    ███████                354 (11.2%)
│   Vikas    █                       24 ( 0.8%)
└───────────────────────────────────────────────────────────

┌─ PER-PERSON STATS ────────────────────────────────────────
│   Name      Media  Deleted    Words  Words/msg
│   Rahul         7        6    2,399        2.6
│   Priya         4        2    3,560        5.0
│   Neha          8        3    3,317        5.3
│   Aman          4        2    2,430 